<a href="https://colab.research.google.com/github/valliansayoga/ey-data-challenge-2025/blob/master/EY2025_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings

warnings.simplefilter(action="ignore", category=pd.errors.PerformanceWarning)
warnings.filterwarnings("ignore")
pd.options.display.max_columns = None

In [3]:
to_drop = ["Latitude", "Longitude", "datetime"]
target = "UHI Index"
solar_stat = "Max"
# df = pd.concat([
#     pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Final.csv"),
#     # pd.read_csv("/content/drive/MyDrive/EY 2025/Train_Building_Features.csv"),
#     # pd.read_csv(f"/content/drive/MyDrive/EY 2025/Train_SolarData{solar_stat}SinceTrainTime.csv"),
# ], axis=1).drop(to_drop, axis=1, errors="ignore")
# feature_shape = df.shape[1]
# feature_shape

In [14]:
from sklearn.feature_selection import SelectPercentile, f_regression, mutual_info_regression
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    VotingRegressor,
    StackingRegressor,
    GradientBoostingRegressor,
    BaggingRegressor,
    AdaBoostRegressor
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from skimage.feature import graycomatrix, graycoprops


def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    insample = r2_score(y_train, model.predict(X_train))
    outsample = r2_score(y_test, model.predict(X_test))
    return insample, outsample

def load_data(typ="Train"):
    global to_drop
    df = pd.concat([
        pd.read_csv(f"/content/drive/MyDrive/EY 2025/{typ}_Final.csv"),
        pd.read_csv(f"/content/drive/MyDrive/EY 2025/{typ}_Building_Features.csv"),
        pd.read_csv(f"/content/drive/MyDrive/EY 2025/{typ}_SolarData{solar_stat}SinceTrainTime.csv")
    ], axis=1).drop(to_drop, axis=1, errors="ignore")
    df = df.pipe(add_features)
    return df

def prepare_data(df, typ, train_size):
    X_train, X_test, y_train, y_test = create_train(df, train_size=train_size)
    return X_train, X_test, y_train, y_test

# def get_glcm(band):
#     band = ((df[f"{band}_median"] - df[f"{band}_min"]) / (df[f"{band}_max"] - df[f"{band}_min"]) * 255).astype(np.uint8)

#     # Define GLCM parameters
#     distances = [10]  # Pixel offsets
#     angles = [0, np.pi/4, np.pi/2, 3*np.pi/4]  # Directions

#     # Compute GLCM
#     glcm = graycomatrix(band, distances=distances, angles=angles, levels=256, symmetric=True, normed=True)

#     # Extract features
#     contrast = graycoprops(glcm, 'contrast').median()
#     entropy = -np.sum(glcm * np.log2(glcm + 1e-10))  # Adding small value to avoid log(0)
#     homogeneity = graycoprops(glcm, 'homogeneity').median()
#     return contrast, entropy, homogeneity

def add_features(df):
    # Existing features
    stats = ["median", "mean", "min", "max", "var", "std"]
    epsilon = 1e-7

    count_cols = df.columns[df.columns.str.contains("count")]
    for col in count_cols:
        divider = int(col.split("_")[0].replace("m", "")[:-1])
        df[f"{col}_density_per_{divider}m"] = df[col] / divider


    # bands = ["red", "green", "blue", "nir08", "swir16", "swir22"]
    # for band in bands:
    #     df[f"{band}_contrast"], df[f"{band}_entropy"], df[f"{band}_homogeneity"] = get_glcm(band)


    for stat in stats:
        df[f"{stat}_evi_x_lwir"] = df[f"evi_{stat}"] * df[f"lwir_{stat}"]
        df[f"{stat}_ndbi_x_lwir"] = df[f"ndbi_{stat}"] * df[f"lwir_{stat}"]
        # df[f"{stat}_ndbi_/_bldg_dnsty"] = df[f"ndbi_{stat}"] / df[f"building_density"].add(epsilon)
        df[f"{stat}_ndbi_/_ndwi"] = df[f"ndbi_{stat}"] / df[f"ndwi_{stat}"].add(epsilon)
        df[f"{stat}_ndbi_/_ndvi"] = df[f"ndbi_{stat}"] / df[f"ndvi_{stat}"].add(epsilon)
        df[f"{stat}_ndbi_/_evi"] = df[f"ndbi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_wvp_/_lwir"] = df[f"wvp_{stat}"] / df[f"lwir_{stat}"].add(epsilon)
        df[f"{stat}_infra_red_combo"] = df[f"nir08_{stat}"] * df[f"swir16_{stat}"] * df[f"swir22_{stat}"]
        df[f"{stat}_relative_ndvi"] = df[f"ndvi_{stat}"] / (df[f"ndvi_{stat}"].max() + epsilon)
        df[f"{stat}_relative_ndwi"] = df[f"ndwi_{stat}"] / (df[f"ndwi_{stat}"].max() + epsilon)
        df[f"{stat}_ndvi_ndwi_ratio"] = df[f"ndvi_{stat}"] / df[f"ndwi_{stat}"].add(epsilon)
        df[f"{stat}_ndvi_evi_ratio"] = df[f"ndvi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_ndwi_evi_ratio"] = df[f"ndwi_{stat}"] / df[f"evi_{stat}"].add(epsilon)
        df[f"{stat}_ndvi_ndwi_diff"] = df[f"ndvi_{stat}"] - df[f"ndwi_{stat}"]
        df[f"{stat}_ndvi_evi_diff"] = df[f"ndvi_{stat}"] - df[f"evi_{stat}"]
        df[f"{stat}_ndwi_evi_diff"] = df[f"ndwi_{stat}"] - df[f"evi_{stat}"]
        df[f"{stat}_combined_spectral_index"] = (df[f"ndvi_{stat}"] + df[f"ndwi_{stat}"] + df[f"evi_{stat}"]) / 3

        df[f"{stat}_bsi"] = (df[f'swir16_{stat}'] + df[f'swir22_{stat}'] - 2 * df[f'nir08_{stat}']) / (df[f'swir16_{stat}'] + df[f'swir22_{stat}'] + 2 * df[f'nir08_{stat}'])
        df[f"{stat}_savi"] = ((df[f'nir08_{stat}'] - df[f'red_{stat}']) * (1 + 0.5)) / (df[f'nir08_{stat}'] + df[f'red_{stat}'] + 0.5)
        df[f"{stat}_sr"] = df[f'nir08_{stat}'] / df[f'red_{stat}']
        df[f"{stat}_dsi"] =  (df[f'swir22_{stat}'] - df[f'nir08_{stat}']) / (df[f'swir22_{stat}'] + df[f'nir08_{stat}'])
        df[f"{stat}_wvi"] = df[f'wvp_{stat}'] / (df[f'swir22_{stat}'] + df[f'swir16_{stat}'] + df[f'nir08_{stat}'])

    df["mean_vci"] = (df[f'ndvi_mean'] - df.ndvi_min) / (df.ndvi_max - df.ndvi_min)
    df["median_vci"] = (df[f'ndvi_median'] - df.ndvi_min) / (df.ndvi_max - df.ndvi_min)

    return df

def create_train(df_features, target="UHI Index", train_size=0.8):
    X = df_features.drop(target, axis=1)
    y = df_features[target]

    X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, train_size=train_size)

    return X_train, X_test, y_train, y_test

def round1(models, train_size, score_func=f_regression, percentile=30):
    to_drop = ["Latitude", "Longitude", "datetime"]
    target = "UHI Index"

    # Round 1 to get pareto + 1 features
    separator = "-"*66
    spaces = " "*24
    equals = "="*32

    print(spaces, "Starting round 1", spaces)
    print(separator)
    df = load_data(typ="Train")
    X_train, X_test, y_train, y_test = prepare_data(df, typ="Train", train_size=train_size)
    # # # # # # # # # # #
    select = SelectPercentile(score_func, percentile=percentile)
    select.fit(X_train, y_train)
    X_train = X_train[select.get_feature_names_out()]
    X_test = X_test[X_train.columns]
    # # # # # # # # # # #

    for model in tqdm(models):
        insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
        model["insample"] = insample
        model["outsample"] = outsample

    results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
    print(equals, "Model Scores", equals)
    print(results)
    print(separator)
    best_model = results.iloc[0]
    print(equals, "Best Model", equals)
    print(best_model.model)
    print(separator)

    if isinstance(best_model.model, (VotingRegressor, StackingRegressor)):
        print("Best model is either voting or stacking!")
        return
    importance = pd.DataFrame(
        {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
    ).sort_values("Importance", ascending=False).reset_index(drop=True)
    importance["cumulative_importance"] = (importance.Importance.cumsum() / importance.Importance.sum()).round(2)
    print(equals, "Feature Importance", equals)
    print(importance)
    print(separator)
    return importance

def round2(models, pareto_threshold, importance, train_size):
    to_drop = ["Latitude", "Longitude", "datetime"]
    target = "UHI Index"
    separator = "-"*66
    spaces = " "*24
    equals = "="*32

    # Round 2 to get pareto + 1
    if importance is not None:
        pareto = importance[importance.cumulative_importance <= pareto_threshold]
        print(equals, "Pareto Features + 1", equals)
        print(pareto)

    print(separator)

    print(spaces, "Starting round 2", spaces)
    print(separator)

    df = load_data(typ="Train")
    # use_cols = [target, *pareto.Features]

    X_train, X_test, y_train, y_test = prepare_data(df, typ="Train", train_size=train_size)
    X_train = X_train[pareto.Features]
    X_test = X_test[pareto.Features]

    for model in tqdm(models):
        # if not isinstance(model["model"], (VotingRegressor, StackingRegressor)):
        #     model["model"].set_params(max_features=n_features)
        insample, outsample = evaluate_model(model["model"], X_train, X_test, y_train, y_test)
        model["insample"] = insample
        model["outsample"] = outsample

    results = pd.DataFrame(models).sort_values("outsample", ascending=False).reset_index(drop=True)
    print(equals, "Model Scores", equals)
    print(results.head(1))
    print(separator)
    best_model = results.iloc[0]
    print(equals, "Best Model", equals)
    print(best_model.model)
    print(separator)

    if isinstance(best_model.model, (VotingRegressor, StackingRegressor)):
        print("Best model is either voting or stacking!")
        return best_model, X_train

    importance = pd.DataFrame(
        {"Features": X_train.columns, "Importance": best_model.model.feature_importances_},
    ).sort_values("Importance", ascending=False).reset_index(drop=True)
    importance["cumulative_importance"] = (importance.Importance.cumsum() / importance.Importance.sum()).round(2)
    print(equals, "Feature Importance", equals)
    print(importance)
    print(separator)
    return best_model, X_train

# Modelling

In [26]:
# max_features = int(feature_shape // 3)
basic_params = {
    "n_jobs": -1,
    "random_state": 0,
    "max_features": "log2"
    # "min_samples_leaf": 5,
}

# vote = VotingRegressor(
#     [
#         ("1", ExtraTreesRegressor(max_features=0.25, n_estimators=200, **basic_params)),
#         ("2", ExtraTreesRegressor(max_features=0.5, n_estimators=200, **basic_params)),
#         ("3", ExtraTreesRegressor(max_features=0.75, n_estimators=200, **basic_params)),
#         ("4", ExtraTreesRegressor(max_features=1.0, n_estimators=200, **basic_params)),
#         ("5", ExtraTreesRegressor(n_estimators=300, **basic_params)),
#     ],
#     n_jobs=-1
# )

# stack = StackingRegressor(
#     [
#         ("1", ExtraTreesRegressor(max_features=0.25, n_estimators=200, **basic_params)),
#         ("2", ExtraTreesRegressor(max_features=0.5, n_estimators=200, **basic_params)),
#         ("3", ExtraTreesRegressor(max_features=0.75, n_estimators=200, **basic_params)),
#         ("4", ExtraTreesRegressor(max_features=1.0, n_estimators=200, **basic_params)),
#     ],
#     ExtraTreesRegressor(n_estimators=300, **basic_params),
#     cv=2,
#     n_jobs=-1
# )

models = [
    {"model": ExtraTreesRegressor(n_estimators=300, **basic_params)},
    {"model": ExtraTreesRegressor(n_estimators=250, **basic_params)},
    {"model": ExtraTreesRegressor(n_estimators=200, **basic_params)},
    # {"model": RandomForestRegressor(n_estimators=100, **basic_params)},
]

importance = round1(
    models,
    train_size=0.9,
    score_func=f_regression,
    percentile=100
)

                         Starting round 1                         
------------------------------------------------------------------


100%|██████████| 3/3 [00:29<00:00,  9.87s/it]


================================ Model Scores ================================
                                               model  insample  outsample
0  (ExtraTreeRegressor(max_features='log2', rando...       1.0   0.968188
1  (ExtraTreeRegressor(max_features='log2', rando...       1.0   0.968061
2  (ExtraTreeRegressor(max_features='log2', rando...       1.0   0.967894
------------------------------------------------------------------
================================ Best Model ================================
ExtraTreesRegressor(max_features='log2', n_estimators=300, n_jobs=-1,
                    random_state=0)
------------------------------------------------------------------
================================ Feature Importance ================================
                     Features  Importance  cumulative_importance
0                   atran_std    0.015480                   0.02
1                max_distance    0.015190                   0.03
2              distance_rang

In [27]:
best_model, X_train = round2(
    models,
    pareto_threshold=0.45,
    importance=importance,
    train_size=0.999
)
# pareto=0.51 | train=0.98 | val=0.977743 | submit=0.9748
# pareto=0.5  | train=0.98 | val=0.977189 | submit=0.9749
# pareto=0.5  | train=0.99 | val=0.972998 | submit=0.9751
# pareto=0.53 | train=0.98 | val=0.980385 | submit=0.9771
# pareto=0.53 | train=0.99 | val=0.980385 | submit=0.9772
# pareto=0.51 | train=0.99 | val=0.977249 | submit=0.9775

================================ Pareto Features + 1 ================================
              Features  Importance  cumulative_importance
0            atran_std    0.015480                   0.02
1         max_distance    0.015190                   0.03
2       distance_range    0.015171                   0.05
3             drad_min    0.014629                   0.06
4      apparent_zenith    0.014571                   0.08
5   apparent_elevation    0.014527                   0.09
6               zenith    0.014255                   0.10
7                  ghi    0.014234                   0.12
8           atran_mean    0.014227                   0.13
9     average_distance    0.014208                   0.15
10            urad_max    0.014143                   0.16
11    airmass_absolute    0.014133                   0.17
12     median_distance    0.013998                   0.19
13            drad_max    0.013772                   0.20
14            drad_var    0.013763          

100%|██████████| 3/3 [00:15<00:00,  5.13s/it]

================================ Model Scores ================================
                                               model  insample  outsample
0  (ExtraTreeRegressor(max_features='log2', rando...       1.0   0.996579
------------------------------------------------------------------
================================ Best Model ================================
ExtraTreesRegressor(max_features='log2', n_estimators=250, n_jobs=-1,
                    random_state=0)
------------------------------------------------------------------
================================ Feature Importance ================================
              Features  Importance  cumulative_importance
0     average_distance    0.040884                   0.04
1      median_distance    0.040276                   0.08
2         max_distance    0.039982                   0.12
3       distance_range    0.036548                   0.16
4   distance_variation    0.033723                   0.19
5     airmass_absolute 

# Predicting Submission

In [28]:
from google.colab import files


def create_submission(filename: str, model):
    global solar_stat
    # sub_df = pd.concat([
    #     pd.read_csv("/content/drive/MyDrive/EY 2025/Submission_Final.csv"),
    #     # pd.read_csv("/content/drive/MyDrive/EY 2025/Submission_Building_Features.csv"),
    #     # pd.read_csv(f"/content/drive/MyDrive/EY 2025/Submission_SolarData{solar_stat}SinceTrainTime.csv"),
    # ], axis=1)
    sub_df = load_data("Submission")

    final_df = pd.read_csv("/content/drive/MyDrive/EY 2025/Submission_Final.csv", usecols=["Latitude", "Longitude"])
    print("Predicting", sub_df.shape[0], "rows...")

    sub_df = add_features(sub_df)

    to_predict = sub_df.loc[:, X_train.columns]
    # print(to_predict.shape)
    # to_predict = sub_df.loc[:, X.columns]

    print("Predicting...")
    final_df["UHI Index"] = model.predict(to_predict)
    final_df.to_csv(filename, index=False)
    print("Done!")
    return

sub_file = f"Rad50m_BacktoOriginalLog2Featrs10.csv"
create_submission(
    sub_file,
    best_model.model
)
files.download(sub_file)

Predicting 1040 rows...
Predicting...
Done!


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!rm *.csv

rm: cannot remove '*.csv': No such file or directory


---